# Tp03 : Étude de cas Yelp

## Création du Dataset

### Import des librairie

In [33]:
from pandas import read_csv, merge, DataFrame
from numpy import nan, floor
from utils import create_data_path, parse_hours
from matplotlib.pyplot import figure, show
from ipywidgets import Dropdown, interact

### Variable

In [34]:
export_path: str = "./restaurants_features.csv"
days: list[str] = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
data_set_path: dict[str, str] = {
    "avis"         : create_data_path("avis.csv"),
    "categories"   : create_data_path("categories.csv"),
    "checkin"      : create_data_path("checkin.csv"),
    "conseils"     : create_data_path("conseils.csv"),
    "horaires"     : create_data_path("horaires.csv"),
    "restaurants"  : create_data_path("restaurants.csv"),
    "services"     : create_data_path("services.csv"),
    "utilisateurs" : create_data_path("utilisateurs.csv")
}

/home/dewk/Code/collecte/tp3
/home/dewk/Code/collecte/tp3
/home/dewk/Code/collecte/tp3
/home/dewk/Code/collecte/tp3
/home/dewk/Code/collecte/tp3
/home/dewk/Code/collecte/tp3
/home/dewk/Code/collecte/tp3
/home/dewk/Code/collecte/tp3


### Charger Dataset

In [35]:
df_avis         = read_csv(data_set_path["avis"])
df_categories   = read_csv(data_set_path["categories"])
df_checking     = read_csv(data_set_path["checkin"])
df_conseils     = read_csv(data_set_path["conseils"])
df_horaires     = read_csv(data_set_path["horaires"])
df_restaurants  = read_csv(data_set_path["restaurants"])
df_services     = read_csv(data_set_path["services"])
df_utilisateurs = read_csv(data_set_path["utilisateurs"])

In [36]:
data = DataFrame(df_restaurants)
data = data.drop(["zone", "ferme"], axis = 1)

In [37]:
review_count = (df_avis
    .groupby("restaurant_id")
    .size()
    .reset_index(name = "review_count_total"))

In [38]:
positive_reviews = (df_avis[df_avis["etoiles"] >= 4]
    .groupby("restaurant_id")
    .size()
    .reset_index(name = "review_positive"))

In [39]:
review = merge(review_count, 
               positive_reviews, 
               left_on= "restaurant_id", 
               right_on="restaurant_id", 
               how="inner")

In [40]:
data = data.merge(review, 
             left_on  = "restaurant_id", 
             right_on = "restaurant_id", 
             how      = "inner")

In [41]:
data["review_count_total"] = (data["review_count_total"]
    .fillna(nan)
    .astype("int64"))

In [42]:
data["review_positive"] = (data["review_positive"]
    .fillna(nan)
    .astype("int64"))

In [43]:
data["positive_ratio"] = floor((data["review_positive"] / data["review_count_total"]) * 100) / 100

In [44]:
checkins_total = (df_checking.groupby("restaurant_id")
    .size()
    .reset_index(name="checkins_total"))

In [45]:
data = data.merge(checkins_total, 
             left_on = "restaurant_id", 
             right_on = "restaurant_id", 
             how = "inner")

In [46]:
data["checkins_total"] = data["checkins_total"].fillna(nan).astype("int64")

In [47]:
name_counts = data["nom"].value_counts()

In [48]:
data["is_chain"] = data["nom"].map(lambda x: name_counts[x] >= 3)

In [49]:
prix_moyen = (df_services
    .groupby("restaurant_id")["prix"]
    .mean()
    .reset_index(name="prix_moyen"))

In [50]:
data = data.merge(prix_moyen, 
                  left_on = "restaurant_id", 
                  right_on = "restaurant_id", 
                  how = "left")

In [51]:
df_elite = df_utilisateurs[df_utilisateurs["elite"].notna()]

In [52]:
df_elite = df_utilisateurs[df_utilisateurs["elite"] != "[]"]

In [53]:
elite_id = df_elite[["utilisateur_id"]]

In [54]:
elite_id

,utilisateur_id
0,l6BmjZMeQD3rDxWUbiAiow
1,4XChL029mKr5hydo79Ljxg
2,bc8C_eETBWL0olvFSJJd0w
3,MM4RJAeH6yuaN8oZDSt0RA
4,TEtzbpgA2BFBrC0y0sCbfw
...,...
748242,qIKRzgQwcLsRbcjFYD1jkg
748243,vyK8txmse-HVk7hAKCvuMQ
748244,gCfFBY-cO1Y0TxxKX89-ig
748245,Z2uEg8DIk1twiNC6NV7e9w


In [55]:
elite_reviews = df_avis.merge(elite_id,
                                on="utilisateur_id",
                                how="inner")


In [56]:
elite_users_count = (elite_reviews
                     .groupby("restaurant_id")["utilisateur_id"]
                     .nunique()
                     .reset_index(name="elite_users_count"))

In [57]:
data = data.merge(elite_users_count,
    on="restaurant_id",
    how="left")

In [58]:
for day in days:
    df_horaires[day] = df_horaires[day].apply(parse_hours)

In [59]:
df_horaires["avg_open_hours"] = df_horaires[days].mean(axis=1)

In [60]:
df_horaires["avg_open_hours"] = floor(df_horaires["avg_open_hours"] * 100) / 100

In [61]:
data = data.merge(df_horaires[["restaurant_id", "avg_open_hours"]],
    on="restaurant_id",
    how="left")

In [62]:
data.to_csv(export_path)

In [63]:
data

,restaurant_id,nom,moyenne_etoiles,ville,review_count_total,review_positive,positive_ratio,checkins_total,is_chain,prix_moyen,elite_users_count,avg_open_hours
0,lCwqJWMxvIUQt1Re_tDn4w,Denny's,2.5,Las Vegas,72,22,0.30,181,True,2.0,70,0.00
1,pd0v6sOqpLhFJ7mkpIaixw,Ike's Love & Sandwiches,4.0,Phoenix,108,82,0.75,492,False,2.0,105,9.14
2,0vhi__HtC2L4-vScgDFdFw,Midori Japanese Cafe,3.5,Calgary,49,33,0.67,157,False,2.0,48,9.64
3,t65yfB9v9fqlhAkLnnUXdg,Pho U,3.5,Toronto,36,21,0.58,18,False,1.0,35,8.14
4,i7_JPit-2kAbtRTLkic2jA,John & Sons Oyster House,4.0,Toronto,88,63,0.71,110,False,3.0,86,7.92
...,...,...,...,...,...,...,...,...,...,...,...,...
31835,cjZfgcQwA6KmQ_ANWKN2aw,Bruegger's Bagels,3.5,McMurray,6,4,0.66,37,True,1.0,6,8.00
31836,Hq2edcOTjse7wjK2CwBijQ,Bistro Pointe-Claire,3.5,Pointe-Claire,11,6,0.54,5,False,2.0,11,9.78
31837,7KlpgRjjAmVabPzxcExs0g,Taco Mex,4.0,Phoenix,11,8,0.72,32,False,1.0,11,0.00
31838,0fY-zYyP2fDmp2YXFsuNTg,Gotham Provisions Company,4.0,Sun Prairie,18,13,0.72,7,False,1.0,18,5.00


In [ ]:
numeric_columns = data.select_dtypes('number').columns

x_dropdown = Dropdown(options=numeric_columns, description='X')
y_dropdown = Dropdown(options=numeric_columns, description='Y')
z_dropdown = Dropdown(options=numeric_columns, description='Z')

city_dropdown = Dropdown(
    options=['(All)'] + sorted(data['ville'].unique()),
    description='Ville'
)

@interact(x=x_dropdown, y=y_dropdown, z=z_dropdown, city=city_dropdown)
def update_plot(x, y, z, city):
    ville = data if city == '(All)' else data[data['ville'] == city]

    fig = figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    ax.scatter(ville[x], ville[y], ville[z], s=40, alpha=0.8)

    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_zlabel(z)
    ax.set_title(f"Scatter 3D : {x} vs {y} vs {z}\nVille : {city}")
    
    show()

interactive(children=(Dropdown(description='X', options=('moyenne_etoiles', 'review_count_total', 'review_posi…